In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py

from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","1_Domain_Profiles"))
from CLASSES_DomainProfiles import DomainProfiles_Class, DomainProfiles_DataLoading_Class

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="Tracking_Algorithms", dataName="Eulerian_CLTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=False

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='1min':
            num_jobs=20
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=100
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
##############################################
#MODEL AND ALGORITHM NUMERICAL PARAMETERS

In [ ]:
kms=np.argmax(ModelData.xh-ModelData.xh[0] >= 1) #finds how many x grids is 1 km
dx=int(ModelData.xh[1]-ModelData.xh[0]) #grid resolution (in km)

In [ ]:
#setting convergence threshold
if ModelData.res=='1km':
    conv_thresh=1.0/1000
elif ModelData.res=="250m":
    conv_thresh=1.5/1000

In [ ]:
##############################################
#DATA LOADING FUNCTIONS

In [ ]:
def GetConvergence(t):
    timeString = ModelData.timeStrings[t]
    varName = 'convergence'
    convergence = CallVariable(ModelData, DataManager, timeString, varName)
    return convergence

In [ ]:
##############################################
#ALGORITHM FUNCTIONS

In [ ]:
#SBZ Convergence Line Search Algorithm (levels are seperate) (python version 3.10.9) (All Max Algorithm)
def FindMaxIndices(yconv):
    """find indices of local maxima above threshold."""

     #takes dconv/dx
    ddx = (yconv[1:  ] - yconv[0:-1]) / (2 * dx)
     
    #finds local max where dconv/dx sign changes
    signs = np.sign(ddx)
    signs_diff=np.diff(signs)
    local_maxes=np.where((signs_diff != 0) & (signs_diff < 0))[0]+1 #make sure +1 is here (it corrects the location of the derivative)
    local_maxes=local_maxes[np.where(yconv[local_maxes]>conv_thresh)] #check if convergence is greater than convergence threshold (1s-1)
    local_maxes=local_maxes[(local_maxes>50*kms)&(local_maxes<len(ModelData.xf)-50*kms)] #removes maxes that are with 50 km of y boundary
    return local_maxes

In [ ]:
#Finds all local maximums (from Calculus) along each y level for a specific z level (~0.28km in this case)
def FindLocalMaxes(conv_z,yind, second_round=False):
    """
    #Find_Local_Maxes Function
    #(1) At a single time and z level, runs through each y-level
    #(2) At each y-level, takes the x-derivative
    #(3) Take sign(x_derivative)
    #(4) Take diff(x_derivative)
    #(5) Max is located one index to the right of where derivative changes from positive to negative or diff is +1
    #[(6) Optional: the algorithm can run a second time over the leftover maxes after removing previous maxes from temporary variable]
    """
    
    #indexes convergence in y
    yconv=conv_z[yind,:]
    
    #finds local max where dconv/dx sign changes
    local_maxes = FindMaxIndices(yconv)

    #second round maxes (not 100% necessary, only if missing many convergence maximums that are visually there)
    if second_round == True:
        yconv2=yconv.copy()
        yconv2[local_maxes]=0
        
        #takes dconv/dx
        local_maxes2 = FindMaxIndicies(f=yconv2)
    
        #combine the two
        local_maxes=np.concatenate((local_maxes,local_maxes2))
    return local_maxes

In [ ]:
##############################################
#RUNNING FUNCTIONS

In [ ]:
def LayerMax(conv): #finds max convergence along y for multiple z location (5 is good)
    
    #initializing data
    Nz = int(np.where(ModelData.zh <= 0.775)[0][-1]) + 1 #only include levels less than 1 km
    maxConvergence_X=np.full((Nz, ModelData.Nyh, ModelData.Nxh), -1, dtype=np.int16)
    
    #running for all z levels
    for zlev in range(Nz):
        #Taking Convergence of current timesftep
        conv_z=conv[zlev,:,:] #current z level for convergence

        for yind in range(ModelData.Nyh): #plot maximums for each row

            #finds all local maxes
            local_maxes=FindLocalMaxes(conv_z,yind) #convergence threshold (in 1/s)
            
            #storing data
            maxConvergence_X[zlev,yind,local_maxes] = local_maxes
    return maxConvergence_X

def GetAvgConvergence(conv,
                      zHeight_km = 2,
                      OPTION="TWO"):
    if OPTION=="ONE":
        return np.mean(conv, axis=(0,1)) 
    elif OPTION=="TWO":
        convMasked = conv.copy()
        zIndex=(np.abs(ModelData.zh - zHeight_km)).argmin()
        convMasked = convMasked[0:zIndex+1]
        convMasked = np.mean(convMasked,axis=0)
        
        x = np.arange(ModelData.Nxh)
        y = np.arange(ModelData.Nyh)
        X, Y = np.meshgrid(x, y)
        mask = (X > 50 * kms) & (X < ModelData.Nxh - 50 * kms)
        
        convMasked[~mask] = np.nan
        return np.nanmean(convMasked, axis=(0))

In [40]:
#Functions for ColdPool_SeaBreeze_Mask
def GetSBFMask(t, xmaxs_t):
    kms = np.argmax(ModelData.xh - ModelData.xh[0] >= 1)
    x = np.arange(len(ModelData.xh))
    y = np.arange(len(ModelData.yh))
    X, Y = np.meshgrid(x, y)
    X = X.astype(float)
    SBF1 = X > xmaxs_t - 10*kms
    SBF2 = X < xmaxs_t + 10*kms
    SBF_mask=SBF1&SBF2
    return SBF_mask

def GetDataAndMasks(t, avgConvergence):
    if t == 0:
        output=np.full((ModelData.Nyh,ModelData.Nxh),False)
        return output,output,output, output,output, output
    else:
        timeString=ModelData.timeStrings[t]
        
        varName = 'theta_v'
        theta_v = CallVariable(ModelData, DataManager, timeString, varName)[0]
        # theta_v_prime = theta_v - np.mean(theta_v) #original
        avgConvergence_max=np.nanmax(avgConvergence); xmaxs_t = np.where(avgConvergence==avgConvergence_max)[0][0]
        # if xmaxs_t + 10 >= ModelData.Nxh-1:
        #     xmaxs_t = int(0.25 * ModelData.Nxh) #if SBF is too close to x-boundary, use ocean instead. (depreceated after update to GetAvgConvergence function)
        theta_v_prime = theta_v - np.mean(theta_v[:,xmaxs_t+10]) #using mean over right of SBF to improve estimate
        theta_v_prime_mask = theta_v_prime < -0.25 
        positive_theta_v_prime_mask = theta_v_prime >= 0
    
        varName = 'qr'
        qr = CallVariable(ModelData, DataManager, timeString, varName)[0]
        qr_mask= qr > 1e-6 
        
        SBF_mask = GetSBFMask(t, xmaxs_t) 
        return theta_v_prime,theta_v_prime_mask,positive_theta_v_prime_mask, qr,qr_mask, SBF_mask
 
def ApplyMaximumFilter(mask,
                       numKilometers = 2):
    radius = int(np.ceil(numKilometers*1000 / ModelData.dx))
    mask_expanded = maximum_filter(mask, size=2*radius+1)
    return mask_expanded
from scipy.ndimage import maximum_filter

def CombineMasks(theta_v_prime_mask,qr_mask,positive_theta_v_prime_mask,SBF_threshold,
                 maxConvergence_X):
    one=ApplyMaximumFilter(theta_v_prime_mask,numKilometers=2)
    two=ApplyMaximumFilter(qr_mask,numKilometers=6)
    three=SBF_threshold
    
    CP_SBF_Mask = (one&two) | three
    
    CP_SBF_Mask_negation = ~ApplyMaximumFilter(CP_SBF_Mask,numKilometers=5) & positive_theta_v_prime_mask
    return np.broadcast_to(CP_SBF_Mask, maxConvergence_X.shape),np.broadcast_to(CP_SBF_Mask_negation, maxConvergence_X.shape)

def ApplyMaskToMaxConvergence(maxConvergence_X, mask):
    maxConvergence_X_output = maxConvergence_X.copy()  # avoid modifying original
    maxConvergence_X_output[~mask] = -1
    return maxConvergence_X_output

In [41]:
def RunAlgorithm():
    for t in tqdm(loop_elements, desc="Processing timesteps"):
        # if t % 1 == 0: print(f'Processing timestep {t}/{len(data["time"])}')

        #getting convergence at time t
        conv=GetConvergence(t)
        # Compute maxConvergence_X for this timestep (z,y,x)
        maxConvergence_X = LayerMax(conv)
        avgConvergence = GetAvgConvergence(conv)

        #calculating ColdPool_SeaBreeze_Mask
        [theta_v_prime,theta_v_prime_mask,positive_theta_v_prime_mask, qr,qr_mask, SBF_mask] = GetDataAndMasks(t, avgConvergence)
        [CP_SBF_Mask,CP_SBF_Mask_negation] = CombineMasks(theta_v_prime_mask,qr_mask,positive_theta_v_prime_mask,SBF_mask,
                                                          maxConvergence_X)

        #applying thresholds to CL data
        maxConvergence_X_CL = ApplyMaskToMaxConvergence(maxConvergence_X,CP_SBF_Mask)
        maxConvergence_X_turbulentCL = ApplyMaskToMaxConvergence(maxConvergence_X,CP_SBF_Mask_negation)

        Dictionary = {"maxConvergence_X": maxConvergence_X,
                      "avgConvergence": avgConvergence,
                      "CP_SBF_Mask": CP_SBF_Mask,
                      "CP_SBF_Mask_negation": CP_SBF_Mask_negation,
                      "maxConvergence_X_CL": maxConvergence_X_CL,
                      "maxConvergence_X_turbulentCL": maxConvergence_X_turbulentCL}

        # Directly write into HDF5 dataset at index t
        timeString = ModelData.timeStrings[t]
        TrackingAlgorithms_DataLoading_Class.SaveData(ModelData,DataManager, Dictionary, timeString)

In [42]:
#############################################
#RUNNING

In [ ]:
RunAlgorithm()

In [43]:
#############################################
#TESTING

In [53]:
def CombineMasks(theta_v_prime_mask,qr_mask,positive_theta_v_prime_mask,SBF_threshold,
                 maxConvergence_X):
    one=ApplyMaximumFilter(theta_v_prime_mask,numKilometers=2)
    # two=qr_mask
    two=ApplyMaximumFilter(qr_mask,numKilometers=6)
    three=SBF_threshold
    
    CP_SBF_Mask = (one&two) | three
    # CP_SBF_Mask = (one) | three
    
    # CP_SBF_Mask_negation = ~CP_SBF_Mask
    # CP_SBF_Mask_negation = ~CP_SBF_Mask & positive_theta_v_prime_mask
    CP_SBF_Mask_negation = ~ApplyMaximumFilter(CP_SBF_Mask,numKilometers=5) & positive_theta_v_prime_mask
    return np.broadcast_to(CP_SBF_Mask, maxConvergence_X.shape),np.broadcast_to(CP_SBF_Mask_negation, maxConvergence_X.shape)

def GetDataAndMasks(t, avgConvergence):
    if t == 0:
        output=np.full((ModelData.Nyh,ModelData.Nxh),False)
        return output,output,output, output,output, output
    else:
        timeString=ModelData.timeStrings[t]
        
        varName = 'theta_v'
        theta_v = CallVariable(ModelData, DataManager, timeString, varName)[0]
        # theta_v_prime = theta_v - np.mean(theta_v) #original
        avgConvergence_max=np.nanmax(avgConvergence); xmaxs_t = np.where(avgConvergence==avgConvergence_max)[0][0]
        # if xmaxs_t + 10 >= ModelData.Nxh-1:
        #     xmaxs_t = int(0.25 * ModelData.Nxh) #if SBF is too close to x-boundary, use ocean instead. (depreceated after update to GetAvgConvergence function)
        theta_v_prime = theta_v - np.mean(theta_v[:,xmaxs_t+10]) #using mean over right of SBF to improve estimate
        theta_v_prime_mask = theta_v_prime < -0.25 #-0.5
        positive_theta_v_prime_mask = theta_v_prime >= 0
    
        varName = 'qr'
        qr = CallVariable(ModelData, DataManager, timeString, varName)[0]
        qr_mask= qr > 1e-6 
        
        SBF_mask = GetSBFMask(t, xmaxs_t) 
        return theta_v_prime,theta_v_prime_mask,positive_theta_v_prime_mask, qr,qr_mask, SBF_mask

def Test(t):
    timeString = ModelData.timeStrings[t]
    print(timeString)
    
    #getting convergence at time t
    conv=GetConvergence(t)
    # Compute maxConvergence_X for this timestep (z,y,x)
    maxConvergence_X = LayerMax(conv)
    avgConvergence = GetAvgConvergence(conv)
    
    #calculating ColdPool_SeaBreeze_Mask
    [theta_v_prime,theta_v_prime_mask,positive_theta_v_prime_mask, qr,qr_mask, SBF_mask] = GetDataAndMasks(t, avgConvergence)
    [CP_SBF_Mask,CP_SBF_Mask_negation] = CombineMasks(theta_v_prime_mask,qr_mask,positive_theta_v_prime_mask,SBF_mask,
                                                      maxConvergence_X)
    
    #applying thresholds to CL data
    maxConvergence_X_CL = ApplyMaskToMaxConvergence(maxConvergence_X,CP_SBF_Mask)
    maxConvergence_X_turbulentCL = ApplyMaskToMaxConvergence(maxConvergence_X,CP_SBF_Mask_negation)
    
    #getting qr 
    varName = 'qr'
    qr = CallVariable(ModelData, DataManager, timeString, varName)[0]
    qr2=qr.copy();qr2[qr2<=1e-6]=np.nan
    
    # plotting
    fig, axes = plt.subplots(1, 2, figsize=(20, 6), sharex=True, sharey=True,gridspec_kw={'wspace': 0.05})
    levels = np.arange(-1.75, 1.75+0.25, 0.25)
    cmap = plt.get_cmap('RdBu_r').copy()
    norm = mcolors.BoundaryNorm(levels, cmap.N)
    
    vmin, vmax = 0, 1  # shared color scale
    
    # fig1
    axis = axes[0]
    d = CP_SBF_Mask
    plot1 = axis.pcolormesh(theta_v_prime,cmap="RdBu_r",norm=norm);plt.colorbar(plot1,label=r"$\theta_v'\ (K)$")
    axis.contour(qr2,colors='red')
    axis.contour(d[3],colors='cyan')
    e = ((maxConvergence_X_CL != -1))
    axis.contour(e[3], colors='green')
    axis.set_title("Cold Pools and Sea Breeze")
    
    # fig2
    axis = axes[1]
    d = CP_SBF_Mask
    plot2 = axis.pcolormesh(theta_v_prime,cmap="RdBu_r",norm=norm);plt.colorbar(plot2,label=r"$\theta_v'\ (K)$")
    axis.contour(qr2,colors='red')
    axis.contour(d[3],colors='cyan')
    e = ((maxConvergence_X_turbulentCL != -1) )
    axis.contour(e[3], colors='green')
    axis.set_title("Turbulent CLs")
    
    # axis labels
    axes[0].set_xlabel("X index");axes[0].set_ylabel("Y index");axes[1].set_xlabel("X index")
    
    # custom legend
    from matplotlib.lines import Line2D
    
    legend_lines = [
        Line2D([0], [0], color='cyan'),
        Line2D([0], [0], color='green'),
        Line2D([0], [0], color='red')
    ]
    
    fig.legend(
        legend_lines,
        ['CP + SBF Threshold', 'CLs','rain > 1e-6'],
        loc='upper center',
        ncol=3)

# Test(t=102)